# Lab 1.1 — Assistant Recon: Where AI Helps, Where It Fails

*Chapter 1 — Introduction to AI in Software Development · 30 minutes · JupyterLab + the OpenAI API via `course_ai`*

Before you hand engineering work to an AI assistant, run a recon: six small,
structured probes — the kinds of tasks you would actually delegate — each scored
**success / partial / fail**, with a note on the *evidence* that decided it. No
vibes: a verdict needs something you checked.

With no API key (or `COURSE_AI_MOCK=1`) the probes run against a deterministic
canned transcript in `course_ai.CANNED_11`. The transcript is deliberately mixed
— four solid answers and two **fluent partial failures** — so the scorecard
mechanics, and your skepticism, both get exercised. With a key, you score the
live model's real answers.

## Objectives

By the end of this lab, you will:

- Probe an AI assistant on six representative engineering tasks.
- Score every answer on evidence you verified, not on how confident it sounded.
- Leave with a personal capability map: what to delegate, what to delegate only
  with verification, and what to keep doing yourself.

## Setup

- **Key:** `OPENAI_API_KEY` from the environment / course `.env`, never printed.
  `COURSE_AI_MOCK=1` (or no key) engages deterministic mock mode.
- **Model:** pinned via `OPENAI_MODEL` (default `gpt-4o-mini`).
- **Mock mode:** the canned transcript answers the probes offline. It is honest
  about being a transcript — but it contains the same *kinds* of mistakes a live
  model makes, planted on purpose.
- **Data rule:** everything here is synthetic. Never paste real proprietary or
  customer code into a prompt without approval.

In [ ]:
import re

import course_ai
from course_ai import chat

print("mode:", course_ai.mode())


def probe(key, prompt):
    """Run one recon probe and print the answer for scoring.

    Live mode: the model answers `prompt`. Mock mode: the deterministic canned
    transcript in course_ai.CANNED_11 stands in, clearly labelled — including
    two deliberate partial failures. You do the scoring either way.
    """
    print(f"===== PROBE {key} =====")
    print("PROMPT>", prompt)
    print()
    if course_ai.MOCK:
        print("[mock transcript — the live model answers this prompt in class]")
        reply = course_ai.CANNED_11[key]
    else:
        reply = chat(prompt)
    print(reply)
    return reply

## How to score a probe

| Verdict | Meaning |
|---|---|
| **success** | You could use the output as-is, after the review you'd give any PR. |
| **partial** | Usable only after you fix or complete something material. |
| **fail** | Wrong, unusable, or confidently fabricated — start over. |

The verdict is about *your ability to use the answer*, not the answer's polish.
Fluent and wrong is **worse** than clumsy and right — fluent-and-wrong slips
past review.

## Steps

Six probes. For each one: read the task, run the probe, **verify what you can
mechanically**, and hold your verdict for the scorecard at the end. About 4–5
minutes per probe.

### Probe 1 — Boilerplate generation (4 min)

The bread-and-butter delegation task: a small utility function with a crisp
contract. Run the probe, then **run the reply's code** in the check cell —
"success" means you could use it as-is after a normal PR review.

In [ ]:
reply1 = probe(
    "boilerplate",
    "Write a Python function `retry(fn, attempts=3, base_delay=0.5)` that calls "
    "fn(), retries on any exception with exponential backoff (the delay doubling "
    "each time), and re-raises the last exception once attempts are exhausted. "
    "Return fn()'s result on success. One code block, no prose.")

In [ ]:
# Verify mechanically: extract the reply's code and run it against a flaky function.
m = re.search(r"```(?:python)?\n(.*?)```", reply1, re.DOTALL)
ns = {}
exec(m.group(1), ns)

calls = {"n": 0}
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("transient failure")
    return "ok"

result = ns["retry"](flaky, attempts=3, base_delay=0)
print("retry() returned:", result, "| attempts used:", calls["n"])
print("contract holds (succeeded on attempt 3):", result == "ok" and calls["n"] == 3)

### Probe 2 — Explain unfamiliar code (4 min)

Handing the assistant something *you* didn't write and can't verify at a glance.
The reply earns **success** only if it names the load-bearing precondition a
caller must satisfy — check that claim against the code yourself.

In [ ]:
UNFAMILIAR = """
import itertools


def _team(r):
    return r["team"]


def by_team(records):
    groups = {}
    for team, grp in itertools.groupby(sorted(records, key=_team), _team):
        groups[team] = list(grp)
    return groups
"""

reply2 = probe(
    "explain",
    "Explain what this Python code does, including any precondition a caller must "
    "satisfy for it to be correct:\n```python" + UNFAMILIAR + "```")

### Probe 3 — Fix a subtle seeded bug (5 min)

This function has exactly one bug, and it is not the obvious one. Run the probe,
then run the check cell and compare the reply's diagnosis with what the code
actually does. Fluent + confident + wrong is the failure mode to internalize.

In [ ]:
SEEDED = """
def moving_average(samples, window=3):
    out = []
    for i in range(len(samples) - window):
        out.append(sum(samples[i:i + window]) / window)
    return out
"""

reply3 = probe(
    "fix_bug",
    "This function has exactly one bug. Find it, explain it, and show the corrected "
    "function:\n```python" + SEEDED + "```")

In [ ]:
# Verify against reality: how many windows SHOULD there be, and how many are there?
ns = {}
exec(SEEDED, ns)
got = ns["moving_average"]([1, 2, 3, 4, 5], 3)
print("seeded function returns:", got)
print("expected three windows — [1,2,3], [2,3,4], [3,4,5] -> averages [2.0, 3.0, 4.0]")
print()
print("Ground truth: the loop bound `len(samples) - window` drops the FINAL window;")
print("the correct bound is `len(samples) - window + 1`.")
print("Did the reply's diagnosis and fix address THAT — or something else? Score it.")

### Probe 4 — A small novel algorithm (6 min)

Nothing in training data matches this spec exactly — and one of its three
constraints is unusual. Run the probe, then let the check cell test the reply's
code against all three constraints, including the one that is easy to miss.

In [ ]:
ALGO_SPEC = """
Write `spread(n, buckets)`: return a list of `buckets` non-negative integers that
1. sums to exactly `n`,
2. differs by at most 1 between any two buckets,
3. is in ASCENDING order — the larger buckets go at the END.

Example: spread(10, 3) -> [3, 3, 4].  One code block, no prose.
"""

reply4 = probe("novel_algo", ALGO_SPEC)

In [ ]:
# Verify all three constraints, on several inputs — not just the spec's example.
m = re.search(r"```(?:python)?\n(.*?)```", reply4, re.DOTALL)
ns = {}
exec(m.group(1), ns)
spread = ns["spread"]

def check_spread(n, k):
    out = spread(n, k)
    ok = {"sums_to_n": sum(out) == n,
          "balanced": max(out) - min(out) <= 1,
          "ascending": out == sorted(out)}
    marks = "  ".join(f"{name}:{'ok' if v else 'MISS'}" for name, v in ok.items())
    print(f"spread({n}, {k}) -> {out}   {marks}")
    return all(ok.values())

results = [check_spread(n, k) for n, k in [(10, 3), (7, 2), (9, 4), (8, 4)]]
print("\nall constraints on all inputs:", all(results))

### Probe 5 — Docstring (4 min)

Documentation drudgery is the assistant's home turf. Score on accuracy: every
section it writes must match what the function actually does.

In [ ]:
CHUNK = """
def chunk(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]
"""

reply5 = probe(
    "docstring",
    "Write a NumPy-style docstring for this Python function; return the function "
    "with the docstring added, one code block:\n```python" + CHUNK + "```")

print()
print("sections present:",
      {"Parameters": "Parameters" in reply5,
       "Yields or Returns": ("Yields" in reply5 or "Returns" in reply5)})

### Probe 6 — Propose test cases (4 min)

Test *ideas*, not test code: the value here is edge coverage. Six cases is the
ask; count them, and ask which important edge is missing.

In [ ]:
PAL = """
def is_palindrome(text):
    cleaned = [c.lower() for c in text if c.isalpha()]
    return cleaned == cleaned[::-1]
"""

reply6 = probe(
    "test_cases",
    "Propose six concrete test cases (input -> expected result) for this function. "
    "Cover the edges you would actually put in CI:\n```python" + PAL + "```")

print()
print("cases proposed:", len(re.findall(r"\n\d+\.", reply6)))

### The scorecard (3 min)

One verdict per probe — **success / partial / fail** — plus the evidence that
decided it. The shipped verdicts below are what the *mock transcript* earns
(yes, two of the canned replies are deliberate partial failures). Running live?
Score what YOUR model produced — disagreement with the shipped card is expected.

In [ ]:
scorecard = {
    # verdict: success | partial | fail   note: the evidence that decided it
    "boilerplate": {"verdict": "success",
                    "note": "code ran as-is; backoff doubled, last exception re-raised"},
    "explain":     {"verdict": "success",
                    "note": "named the sorted-input precondition groupby relies on"},
    "fix_bug":     {"verdict": "partial",
                    "note": "confident misdiagnosis — blamed the division, missed the "
                            "range bound; its own example output was wrong"},
    "novel_algo":  {"verdict": "partial",
                    "note": "ran, sums and balances — but larger buckets land FIRST, "
                            "violating the ascending-order constraint"},
    "docstring":   {"verdict": "success",
                    "note": "Parameters/Yields/Raises all match the real function"},
    "test_cases":  {"verdict": "success",
                    "note": "six cases incl. empty string, case-fold, punctuation"},
}
# ^ these are the verdicts the MOCK TRANSCRIPT earns. In live mode — or if you
#   disagree — overwrite them with your own before running the summary.

In [ ]:
def summarize(card):
    VALID = ("success", "partial", "fail")
    width = max(len(k) for k in card)
    counts = {v: 0 for v in VALID}
    print(f"{'probe':{width}}  verdict   note")
    print(f"{'-' * width}  -------   ----")
    for k, v in card.items():
        assert v["verdict"] in VALID, f"{k}: verdict must be one of {VALID}"
        counts[v["verdict"]] += 1
        print(f"{k:{width}}  {v['verdict']:<9} {v['note']}")
    print(f"\nsuccess {counts['success']} | partial {counts['partial']} | fail {counts['fail']}")
    print("\ncapability map:")
    for verdict, label in [("success", "delegate freely         "),
                           ("partial", "delegate + verify first "),
                           ("fail", "keep doing yourself     ")]:
        print(f"  {label}: {[k for k, v in card.items() if v['verdict'] == verdict]}")

summarize(scorecard)

## Deliverable

1. Your completed `scorecard` — six verdicts, each with a one-line evidence note.
2. The printed summary table and capability map.
3. One sentence: your personal rule for what you will delegate to an assistant
   this month — and what you will verify first.

## Reflection

1. Which probe result surprised you most — and would you have predicted it?
2. The two partials were fluent and confident. What *mechanically checkable*
   property caught each one before the prose could talk you into it?
3. Pick one task from your current sprint: where does it land on your map —
   delegate, delegate-with-verification, or keep?

## Debrief (instructor-led)

1. Compare scorecards: where did the class disagree on a verdict? What evidence
   settled it?
2. Probe 4's reply ran without errors. Why is "it runs" not evidence of
   correctness? (Link forward: Lab 5.1 makes the tests the referee.)
3. Which of the six tasks is safest to delegate *without* verification — and is
   the honest answer ever "none"?
4. Bridge to Lab 2.1: some failures are worse than wrong code — invented APIs,
   invented facts, skewed wording. Next lab audits those.

## Troubleshooting

- **Every probe prints the mock transcript** — no key is visible, or
  `COURSE_AI_MOCK=1` is set. On the classroom VM the key is injected; the lab is
  fully usable either way.
- **The live model aced a probe the transcript flunks** — expected: models vary,
  the transcript is fixed by design. Score what *you* got.
- **`ModuleNotFoundError: course_ai`** — the kernel's working directory is not
  `labs/`. Restart the kernel from the `labs/` folder and Run All.
- **`AttributeError` in a check cell after a live reply** — the model answered in
  prose instead of a code block; add "one code block, no prose" to your prompt
  and re-run the probe.
- **A reply is truncated** — raise `max_tokens` in the `chat()` call inside
  `probe()`.